<a href="https://colab.research.google.com/github/Abstract-code-ops/megascolia-maculata-detector/blob/main/Wasp_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyngrok ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.2 MB/s eta 0:00:00


In [2]:
!unzip "wasp app.zip"
!unzip "best_openvino_model.zip"

Archive:  wasp app.zip
   creating: wasp_app/
   creating: wasp_app/static/
   creating: wasp_app/static/processed/
  inflating: wasp_app/README.md      
   creating: wasp_app/templates/
  inflating: wasp_app/templates/index.html  
  inflating: wasp_app/CPU_SPEED_FIX.py  
  inflating: wasp_app/app.py         
  inflating: wasp_app/requirements_extras.txt  
  inflating: wasp_app/db.py          
Archive:  best_openvino_model.zip
  inflating: metadata.yaml           
  inflating: best.bin                
  inflating: best.xml                


In [3]:
import os, shutil, zipfile
from pathlib import Path
from pyngrok import ngrok
from google.colab import userdata

# ── 1. Config — edit these two paths for your session ──────────────────────
APP_DIR = Path("/content/wasp_app")               # where your wasp_app files live (app.py, db.py, templates/)
OPENVINO_SOURCE = Path("/content/best_openvino_model.zip")   # folder OR .zip — script detects which
# ─────────────────────────────────────────────────────────────────────────────

DEST_NAME = OPENVINO_SOURCE.stem
if not DEST_NAME.endswith('_openvino_model'):
  DEST_NAME += '_openvino_model'

dest_path = APP_DIR / DEST_NAME

# Wipe any stale copy so old weights never get silently reused
if dest_path.exists():
    shutil.rmtree(dest_path)

if OPENVINO_SOURCE.suffix == ".zip":
    print(f"[INFO] Unzipping {OPENVINO_SOURCE} -> {dest_path}")
    with zipfile.ZipFile(OPENVINO_SOURCE, "r") as zf:
        names= zf.namelist()
        top_level_dirs= {n.split("/n")[0] for n in names if "/" in n}
        wrapped = len(top_level_dirs) == 1 and all(n.startswith(f"{list(top_level_dirs)[0]}/") for n in names)

        if wrapped:
            zf.extractall(APP_DIR)
            extracted_folder = APP_DIR / list(top_level_dirs)[0]
            if dest_path.exists():
                shutil.rmtree(dest_path)
            extracted_folder.rename(dest_path)
        else:
            # Loose files at the zip root — extract straight into dest_path
            dest_path.mkdir(parents=True, exist_ok=True)
            zf.extractall(dest_path)

elif OPENVINO_SOURCE.is_dir():
    print(f"[INFO] Copying folder {OPENVINO_SOURCE} -> {dest_path}")
    shutil.copytree(OPENVINO_SOURCE, dest_path)
else:
    raise FileNotFoundError(f"OPENVINO_S.zip/folder: {OPENVINO_SOURCE}")

# Sanity check — an OpenVINO export should contain a .xml + .bin pair
xml_files = list(dest_path.glob("*.xml"))
if not xml_files:
    raise RuntimeError(f"No .xml file found in {dest_path} — this doesn't look like a valid OpenVINO export.")
print(f"[INFO] Verified OpenVINO model files: {[f.name for f in dest_path.iterdir()]}")

# ── 2. Point the app at it ───────────────────────────────────────────────────
os.environ["MODEL_PATH"] = str(dest_path)
print(f"[INFO] MODEL_PATH set to: {os.environ['MODEL_PATH']}")


[INFO] Unzipping /content/best_openvino_model.zip -> /content/wasp_app/best_openvino_model
[INFO] Verified OpenVINO model files: ['metadata.yaml', 'best.bin', 'best.xml']
[INFO] MODEL_PATH set to: /content/wasp_app/best_openvino_model
[INFO] App will be live at: NgrokTunnel: "https://doze-computer-dragging.ngrok-free.dev" -> "http://localhost:5000"
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] Running on [CPU]
[INFO] Loading model from: /content/wasp_app/best_openvino_model
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
[INFO] Model successfully loaded.
[INFO] Detection log databa

In [16]:
!pkill -f "python app.py"

In [17]:
# ── 3. ngrok tunnel + serve ───────────────────────────────────────────────────
NGROK_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)
print(f"[INFO] App will be live at: {public_url}")

os.chdir(APP_DIR)
!python app.py

[INFO] App will be live at: NgrokTunnel: "https://doze-computer-dragging.ngrok-free.dev" -> "http://localhost:5000"
[INFO] Running on [CPU]
[INFO] Loading model from: /content/wasp_app/best_openvino_model
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
[INFO] Model successfully loaded.
[INFO] Detection log database ready at: /content/wasp_app/wasp_detections.db
[INFO] FRAME_SKIP_RATE = 10
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
127.0.0.1 - - [19/Jul/2026 14:29:35] "GET / HTTP/1.1" 200 -
frame---1 detection started
Loading /content/wasp_app/best_openvino_model for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...
frame---1 detection finished
frame---11 detection started
frame---11 

In [18]:
!zip -r output_file.zip /content/wasp_app

  adding: content/wasp_app/ (stored 0%)
  adding: content/wasp_app/best_openvino_model/ (stored 0%)
  adding: content/wasp_app/best_openvino_model/metadata.yaml (deflated 35%)
  adding: content/wasp_app/best_openvino_model/best.bin (deflated 16%)
  adding: content/wasp_app/best_openvino_model/best.xml (deflated 94%)
  adding: content/wasp_app/requirements_extras.txt (deflated 8%)
  adding: content/wasp_app/static/ (stored 0%)
  adding: content/wasp_app/static/processed/ (stored 0%)
  adding: content/wasp_app/static/processed/temp_out_1784469280.mp4 (deflated 9%)
  adding: content/wasp_app/static/processed/out_1784467551.mp4 (deflated 0%)
  adding: content/wasp_app/static/processed/out_1784471029.mp4 (deflated 47%)
  adding: content/wasp_app/static/processed/out_1784468689.mp4 (deflated 47%)
  adding: content/wasp_app/static/processed/temp_out_1784468642.mp4 (deflated 47%)
  adding: content/wasp_app/static/processed/out_1784470727.mp4 (deflated 47%)
  adding: content/wasp_app/static/pro

In [19]:
from google.colab import files

files.download('output_file.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>